# Author: José Ángel de Bustos Pérez
# License: GNU General Public License v3.0

# Basic implementation for CKKS algorithm using Pyphel (https://pyfhel.readthedocs.io/)

CKKS (Cheon–Kim–Kim–Song, 2017) is a homomorphic encryption scheme based on RLWE (Ring Learning With Errors), like BFV, but designed to work with real or complex numbers in an approximate way. That means that the operations will add some error.

Key differences compared to BFV:

* Data: float/double, not integers.
* Results: approximate (there is rounding error).
* Rescaling: after each multiplication, rescaling is required.
* Levels: each rescale consumes a level of the ciphertext.
* Slots: n/2 instead of n (slots are complex conjugates).

It is the standard scheme for machine learning on encrypted data, statistics, signal processing, etc.

The first step is to set the parameters and generate the keys:

* **n**, polynomial degree (power of 2). In CKKS, the number of slots is n/2.
* **scale**, scaling factor. CKKS encodes floats by multiplying them by this factor to convert them into large integers. A larger scale gives more decimal precision but adds more noise. Typical scale value used: 2^30.
* **qi_sizes**, sizes (in bits) of the prime moduli that form the modulus chain. CKKS uses modulus switching: each rescale after a multiplication removes one prime from the chain. The number of intermediate primes equals the multiplicative depth you can support.
* **sec**, security level in bits.

In [1]:
from Pyfhel import Pyfhel
import numpy as np

HE = Pyfhel()

ckks_params = {
    'scheme': 'CKKS',
    'n': 2**14,                # 16384 coefficients => 8192 slots for batching
    'scale': 2**30,            # approximate precision: ~9 decimal places
    # qi_sizes gives a chain of 7 primes. The first and last are “large” (60 bits) for technical reasons; 
    # the ones in the middle are the size of the scale (30 bits). ⇒ Approximately 5 levels of multiplication.
    'qi_sizes': [60, 30, 30, 30, 30, 30, 60],
    'sec': 128,
}

HE.contextGen(**ckks_params)
HE.keyGen()           # Key creation
HE.rotateKeyGen()     # Required for rotations (we will use it in batching)
HE.relinKeyGen()      # Required to reduce size after multiplication

print(f"Scheme: {HE.scheme}")
print(f"n (Polynomial degree): {HE.n}")
print(f"Available Slots (n/2): {HE.n // 2}")
print(f"Scale: 2^{int(np.log2(HE.scale))}")
print(f"qi modulus chain: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

Scheme: Scheme_t.ckks
n (Polynomial degree): 16384
Available Slots (n/2): 8192
Scale: 2^30
qi modulus chain: [60, 30, 30, 30, 30, 30, 60]
=> Available multiplicative depth: 5


Unlike BFV, in CKKS the **noise budget** is not measured in the same way. What matters is the LEVEL of the ciphertext (how many primes remain in the modulus chain). Each rescale consumes one level. When you reach the last level, you can no longer perform multiplications.

## Basic operations

We will perform basic operations such as addition and multiplications to check how noise is increasing. We will start with addition:

In [2]:
import random

# Randon number generation
a = random.random()
b = random.random()

expected_value = a + b

# Encrypting data. CKKS always works with vectors. To encrypt a scalar, we place it
# in an array (the remaining slots are filled with zeros).
fhe_a = HE.encryptFrac(np.array([a], dtype=np.float64))
fhe_b = HE.encryptFrac(np.array([b], dtype=np.float64))

# Homomorphic operation (addition)
fhe_suma = fhe_a + fhe_b

# Getting the operation value, decrypting
suma = HE.decryptFrac(fhe_suma)[0]

# Error
error = abs(expected_value - suma)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic addition: {suma}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.4369626452247829, b = 0.4790811675513129
Expected value: 0.9160438127760958
Homomorphic addition: 0.9160396932184689
Error: 4.12e-06


We are going to proceed with multiplication.

In [3]:
expected_value = a * b

# Homomorphic multiplication
fhe_mult = fhe_a * fhe_b
# Realinarization
~fhe_mult

# Rescalation. After each multiplication, we must manually rescale. CKKS multiplies two numbers scaled by 
# S, and the result ends up scaled by S^2. The rescale operation divides by S and removes one prime from the 
# chain to keep the original scaling factor S
HE.rescale_to_next(fhe_mult)

# Getting the operation value, decrypting
mult = HE.decryptFrac(fhe_mult)[0]

# Error
error = abs(expected_value - mult)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic multiplication: {mult}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.4369626452247829, b = 0.4790811675513129
Expected value: 0.20934057425059913
Homomorphic multiplication: 0.20933193667306593
Error: 8.64e-06


## Performing homomorphic encryption with multiple operations

We are going to show how to perform more complex operations with homomorphic encryption. Let's assume we want to perform homomorphic encryption to:

$$f(x,y) = (x+y)^2 - (x-y)^2$$

In [4]:
# Random number generation
x = random.random()
y = random.random()

expected_value = (x + y)**2 - (x-y)**2

# Encrypting data
fhe_x = HE.encryptFrac(np.array([x], dtype=np.float64))
fhe_y = HE.encryptFrac(np.array([y], dtype=np.float64))

# (x + y)
fhe_addition_xy = fhe_x + fhe_y

# (x + y)^2
fhe_square_add = fhe_addition_xy * fhe_addition_xy
# Realinarization
~fhe_square_add
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_add)

# (x - y)
fhe_substraction_xy = fhe_x - fhe_y

# (x - y)^2
fhe_square_substraction = fhe_substraction_xy * fhe_substraction_xy
# Realinarization
~fhe_square_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_substraction)

# Final. Before doing the substraction both ciphertext must have the same
# level, in both we have performed one rescale operation so they are at the
# same level.
fhe_final = fhe_square_add - fhe_square_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**2")
print(f"Expected value: {expected_value}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.07412668924681964 + 0.320899337224002)**2 - (0.07412668924681964-0.320899337224002)**2
Expected value: 0.09514882179965592
Homomorphic value: 0.09514154422466325
Error: 7.28e-06


Added noise is mostly the same that the noise added by the homomorphic multiplication. Now, we will perform the following:

$$f(x,y) = (x+y)^2 - (x-y)^3$$

In this example $(x+y)^2$ has a multiplicativity depth of 1 but $(x-y)^3$ has a multiplicativity depth of 2. That means that we will need to align levels (working on the same scale) to operate with them.

In [5]:
expected_value_third = (x + y)**2 - (x - y)**3

# Aligning data on the same scale. fhe_substraction_xy is in level 1
# but fhe_square_substraction is in level 2
fhe_substraction_xy_aligned = fhe_substraction_xy.copy()
HE.mod_switch_to_next(fhe_substraction_xy_aligned)

# (x - y)^3
fhe_third_substraction = fhe_square_substraction * fhe_substraction_xy_aligned
# Realinarization
~fhe_third_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_third_substraction)

# (x + y)^2 is in level 1 (one rescaling) but (x - y)^3 is in level 2
# before operating them, we need to have both of them on the  same level
fhe_square_add_aligned = fhe_square_add.copy()
HE.mod_switch_to_next(fhe_square_add_aligned)

# Final
fhe_final = fhe_square_add_aligned - fhe_third_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value_third - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**3")
print(f"Expected value: {expected_value_third}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.07412668924681964 + 0.320899337224002)**2 - (0.07412668924681964-0.320899337224002)**3
Expected value: 0.17107321132040068
Homomorphic value: 0.17109260198460946
Error: 1.94e-05


## Batching operation

Batching allows us to perform the same homomorphic operation to several data at the same time, parallelism.

If we are familiar with processor architectures we will know what **SIMD** (**S**imple **I**nstruction **M**ultiple **D**ata) is. One single operation using only one clock cicle operate on multiple data. Batching is the same but used in homomorphic encryption and it is used to speed up operations.

In [6]:
# Randon number generation
size = 10 # number of floats to operate at the same time
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypting data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Item by item homomorphic addition
fhe_add_vectorial = fhe_v1 + fhe_v2

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_add_vectorial)[:size]

# Error
expected_value = v1 + v2
error = np.abs(final - expected_value)

print(f"v1: {v1}")
print(f"v2: {v2}")
print(f"Homomorphic v1 + v2:  {final}")
print(f"Expected value: {expected_value}")
print(f"Error: {error}")

v1: [ 74.47314531  50.4991106  -55.77722642 -83.62967136 -69.03921199
 -86.38336227  -1.29149844  -1.38818971 -19.83321096  86.57298839]
v2: [ 81.69575282  -1.56622605   1.34706055 -13.61420155  -7.47016371
 -50.44478119 -99.51998867   2.13631792  -6.54980002  78.76487755]
Homomorphic v1 + v2:  [ 156.16889619   48.93289134  -54.43016303  -97.24387733  -76.50938
 -136.82814107 -100.81148529    0.74812358  -26.38301083  165.33786506]
Expected value: [ 156.16889813   48.93288456  -54.43016587  -97.2438729   -76.5093757
 -136.82814346 -100.81148711    0.74812821  -26.38301098  165.33786594]
Error: [1.93311851e-06 6.78784531e-06 2.83808343e-06 4.43129906e-06
 4.29694487e-06 2.39009435e-06 1.82083619e-06 4.63568967e-06
 1.50848443e-07 8.83799146e-07]


Let's see what happens with batching multiplication:

In [7]:
# Item by item homomorphic multiplication
fhe_multiplication_vectorial = fhe_v1 * fhe_v2
# Realinarization
~fhe_multiplication_vectorial

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_multiplication_vectorial)[:size]

# Error
expected_value = v1 * v2
error = np.abs(final - expected_value)

print(f"\nHomomorphic v1 * v2  = {final}")
print(f"Expected value = {v1 * v2}")
print(f"Error: {error}")


Homomorphic v1 * v2  = [ 6.08413953e+03 -7.90927923e+01 -7.51353828e+01  1.13855148e+03
  5.15734213e+02  4.35758967e+03  1.28529787e+02 -2.96561431e+00
  1.29903562e+02  6.81891075e+03]
Expected value = [ 6.08413967e+03 -7.90930225e+01 -7.51353013e+01  1.13855120e+03
  5.15734216e+02  4.35758981e+03  1.28529910e+02 -2.96561456e+00
  1.29903566e+02  6.81891083e+03]
Error: [1.43325778e-04 2.30202077e-04 8.15331484e-05 2.74963571e-04
 3.24069424e-06 1.36127233e-04 1.23130800e-04 2.54697222e-07
 3.94468972e-06 7.58423339e-05]


## Dot product or scalar product

Dot product or scalar product is a fundamental operation used in multiple algorithms such as linear regresion, support vector machines (SVM) or neural network algorithms. For this reason been able to operate it using homomorphic encription will ease using such algoritms with homomorphic encryption.

In [8]:
# Random sample
size = 10 # random sample size
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypt data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Cdot operation with encrypted data
fhe_cdot = fhe_v1 * fhe_v2
# Relinearization
~fhe_cdot

# vector with product, component to component
value_cdot = HE.decryptFrac(fhe_cdot)[:size]

# Error
expected_value = float(np.dot(v1, v2))
error = abs(expected_value - np.sum(value_cdot))

print(f"v1: {v1}")
print(f"v2: {v2}")
# cdot product is the sum for all the values in value_cdot
print(f"Homomorphic v1 * v2: {np.sum(value_cdot)}")
print(f"Expected value: {expected_value}")
print(f"Error: {error:.2e}")

v1: [-63.31625672  86.01025874  82.48331423   9.30615505  72.83766155
 -15.04314985  33.28746635  97.57373577 -33.52430515 -22.52281138]
v2: [ 70.53514842   4.85017008  -5.98100071   0.26969727  33.056316
 -77.44891147  66.70056288  69.75035216  86.96837929 -18.13281968]
Homomorphic v1 * v2: 5552.08391651033
Expected value: 5552.083005277801
Error: 9.11e-04


## How many operations can be done before data is corrupted?

We have seen that the multiplication operation adds noise to the data. For this reason is important to know how many operations can be done without corrupting data. This number  of operations is the operational limit which indicates the maximum number of operations that can be performed by the algorithm.

In CKKS the limit is the prime number chain defined in **qi_sizes = [60, 30, 30, 30, 30, 30, 60]**. With each rescaling operation one prime number is used, in this case we have five multiplication levels

In [9]:
from random import randrange

data = randrange(10)
fhe_data = HE.encryptFrac(np.array([data], dtype=np.float64))

print(f"Encrypting data = {data}")
print(f"Initial qi_sizes: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

# Start multiplying encrypted data
for i in range(1, 10):
    try:
        fhe_data = fhe_data * fhe_data   
        # Relinearization
        ~fhe_data
        # Rescaling
        HE.rescale_to_next(fhe_data)
        expected_value = data ** (2 ** i)
        fhe_value = HE.decryptFrac(fhe_data)[0]
        error = abs(expected_value - fhe_value)
        print(f"Iteration {i}: data^{2**i:<5} | "
              f"Expected value={expected_value:<20.6f} "
              f"Encrypted value={fhe_value:<20.6f} | "
              f"Error={error:.2e}")
    except Exception as e:
        print(f"\nIteration {i}: FALLO - no more multiplication levels available.")
        print(f"  Excepción: {type(e).__name__}: {e}")
        break

Encrypting data = 2
Initial qi_sizes: [60, 30, 30, 30, 30, 30, 60]
=> Available multiplicative depth: 5
Iteration 1: data^2     | Expected value=4.000000             Encrypted value=3.999995             | Error=5.23e-06
Iteration 2: data^4     | Expected value=16.000000            Encrypted value=15.999959            | Error=4.13e-05
Iteration 3: data^8     | Expected value=256.000000           Encrypted value=255.998683           | Error=1.32e-03
Iteration 4: data^16    | Expected value=65536.000000         Encrypted value=65535.325672         | Error=6.74e-01
Iteration 5: data^32    | Expected value=4294967296.000000    Encrypted value=4294878909.589251    | Error=8.84e+04

Iteration 6: FALLO - no more multiplication levels available.
  Excepción: ValueError: scale out of bounds
